# Model Deployment Pipeline

Create an improved model version, place it into shadow mode, and prepare promotion criteria.

In [ ]:
import os
from clario360 import Client

sdk = Client.from_env()
MODEL_NAME = os.environ.get('CLARIO360_MODEL_NAME', 'cyber-sigma-evaluator')
PROMOTE_IF_READY = os.environ.get('CLARIO360_PROMOTE_IF_READY', 'false').lower() == 'true'
ROLLBACK_IF_DIVERGENCE = os.environ.get('CLARIO360_ROLLBACK_IF_DIVERGENCE', 'false').lower() == 'true'
SHADOW_ROLLBACK_REASON = os.environ.get('CLARIO360_ROLLBACK_REASON', 'shadow divergence exceeded threshold')
model = sdk.ai.models.get_by_name(MODEL_NAME)
shadow_config = {'threshold': 0.55, 'source': 'notebook-shadow-candidate'}
shadow_metrics = {'precision': 0.82, 'recall': 0.79, 'f1': 0.805}


## Create a governed version

The registry creates an immutable version so later predictions remain traceable to a fixed configuration.

In [ ]:
version = sdk.ai.models.create_version(model.id, config=shadow_config, metrics=shadow_metrics, lifecycle_stage='shadow')
print(version)
sdk.ai.lifecycle.shadow(model.id, 'start', getattr(version, 'id', version.get('id')))

## Promotion workflow

After shadow comparison data is available, use the lifecycle APIs to promote or roll back.

In [ ]:
version_id = getattr(version, 'id', version.get('id'))
if PROMOTE_IF_READY:
    promoted = sdk.ai.lifecycle.promote(model.id, version_id)
    print(promoted)
elif ROLLBACK_IF_DIVERGENCE:
    rollback = sdk.ai.lifecycle.rollback(model.id, SHADOW_ROLLBACK_REASON)
    print(rollback)
else:
    print('Shadow version created and started. Set CLARIO360_PROMOTE_IF_READY=true to promote or CLARIO360_ROLLBACK_IF_DIVERGENCE=true to roll back in a later run.')